# 03 — Betesmarksklassificering: Munkarp, Höörs kommun (Skåne)

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_040-Grazing-Munkarp.ipynb)

**Syfte:** Automatisera kontroll av EU-jordbruksstöd (CAP) — identifiera om deklarerade betesmarker (LPIS-block) faktiskt används till bete under växtsäsongen, eller om de är övergivna / använda till annat.

**Partners som bidragit:**
- **Jordbruksverket (SJV)** — LPIS-data (Land Parcel Identification System)
- **Naturvårdsverket (SEPA)** — NMD (Nationella Marktäckedata) referens
- **Skogsstyrelsen** — skogsavgränsning
- **RISE** — PIB Grazing CNN-biLSTM-modell, ImintEngine

**Datakällor:**
- Copernicus Sentinel-2 L2A (19 datum under växtsäsongen 2025) via Digital Earth Sweden
- LPIS 2025 (Jordbruksverket) — blockpolygoner
- NMD 2018 (Naturvårdsverket) — basklasser
- PIB ML Grazing CNN-biLSTM (RISE, 2025) — tidsserieklassificering

**Licens:** CC0 1.0 (notebook) · LPIS öppna data via SJV · NMD öppen data via NV

## 1. Metod

**Pipeline:**

```
LPIS-block → NDVI-tidserie per polygon (19 datum)
          → CNN-biLSTM klassificering
             • active_grazing (faktiskt bete — typisk NDVI-sänkning sommartid)
             • no_activity (övergivet / inget bete)
             • error (osäker klassificering)
          → Korsreferens mot NMD (är det verkligen jordbruksmark?)
```

**PIB-modellen** (Preliminary Inspection by Boundary):
- CNN-backbone → extraherar spatiala features per datum
- biLSTM-head → modellerar temporala mönster över växtsäsongen
- Tränad på SJV:s fältkontrollerade block 2021–2024

**Referensresultat för showcase-AOI** (Munkarp, 80 LPIS-block):
- 68 aktivt betade (85%)
- 8 utan aktivitet (10%)
- 4 klassificeringsfel (5%)
- Medelkonfidens: 0.73

## 2. Setup

In [ ]:
# Verified API helper — wraps IMINTResult access
def get_outputs(result, name):
    """Return outputs dict for first successful analyzer matching `name`, else None."""
    for ar in result.analyzer_results:
        if ar.analyzer == name and ar.success:
            return ar.outputs
    return None


In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import folium
import json

# Munkarp, Höörs kommun — showcase-AOI med 80 LPIS-block
AOI = {
    "west": 13.42,
    "south": 55.935,
    "east": 13.48,
    "north": 55.965,
}
DATE = "2025-06-14"  # Sommardatum — tydlig bete-signatur
YEAR = 2025

print(f"AOI (Munkarp): {AOI}")
print(f"Referensdatum: {DATE}")
print(f"LPIS-år: {YEAR}")

## 3. Kör analys

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/bete_lund",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

# Summera grazing-klassificering
if get_outputs(result, "grazing") is not None:
    grazing = get_outputs(result, "grazing").data
    preds = grazing.get("predictions", {})
    print(f"LPIS-block analyserade: {preds.get('total_polygons', 0)}")
    print(f"  Aktivt bete:     {preds.get('active_grazing', 0)}")
    print(f"  Ingen aktivitet: {preds.get('no_activity', 0)}")
    print(f"  Fel/osäker:      {preds.get('errors', 0)}")
    print(f"  Medelkonfidens:  {preds.get('mean_confidence', 0):.2f}")

## 4. Visualisering

In [ ]:
# Interaktiv karta — LPIS-block färgade efter bete-klassificering
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri",
    name="Satellit",
).add_to(m)

color_map = {
    "active_grazing": "#2ca02c",
    "no_activity": "#d62728",
    "error": "#ff7f0e",
}

if get_outputs(result, "grazing") is not None:
    polygons = get_outputs(result, "grazing").data.get("polygons", [])
    for poly in polygons:
        cls = poly.get("class", "error")
        folium.GeoJson(
            poly.get("geometry"),
            style_function=lambda x, c=cls: {
                "fillColor": color_map.get(c, "#888"),
                "color": color_map.get(c, "#888"),
                "weight": 1.5,
                "fillOpacity": 0.4,
            },
            popup=f"{cls} · conf {poly.get('confidence', 0):.2f}",
        ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Statiska paneler: RGB + NDVI + NMD-overlay + LPIS-overlay
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(result.rgb)
axes[0, 0].set_title(f"RGB ({DATE})")
axes[0, 0].axis("off")

if get_outputs(result, "spectral") is not None and "ndvi" in get_outputs(result, "spectral").data:
    ndvi = get_outputs(result, "spectral").data["ndvi"]
    im = axes[0, 1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    axes[0, 1].set_title("NDVI — vegetationsvigor")
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046)
axes[0, 1].axis("off")

if get_outputs(result, "nmd") is not None and "overlay" in get_outputs(result, "nmd").data:
    axes[1, 0].imshow(get_outputs(result, "nmd").data["overlay"])
    axes[1, 0].set_title("NMD landcover-overlay")
axes[1, 0].axis("off")

if get_outputs(result, "grazing") is not None and "lpis_overlay" in get_outputs(result, "grazing").data:
    axes[1, 1].imshow(get_outputs(result, "grazing").data["lpis_overlay"])
    axes[1, 1].set_title("LPIS-block färgkodade efter bete-klassificering")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning

**Operativ användning:**

Jordbruksverket måste enligt EU-regelverket kontrollera deklarerade arealstöd. Traditionellt sker detta via:
1. Fältinspektion (~2% av blocken) — dyrt, begränsad täckning
2. Administrativ kontroll av papper — fångar inte markanvändning

Satellitbaserad klassificering (PIB) erbjuder **100% täckning** — alla LPIS-block klassificeras varje år. Endast avvikande fall behöver fältinspektion.

**Viktigt att notera:**
- Modellen är **beslutsstöd**, inte automatiskt beslutsunderlag
- Felklassificering 5% → kräver fortsatt fältvalidering för stickprov
- Tidig skörd, torka, extrema väderhändelser kan skapa falska "no activity"-klassificeringar

**Koppling till Agenda 2030:**
- SDG 2 (Ingen hunger) — säkra livsmedelsproduktion
- SDG 15 (Ekosystem och biologisk mångfald) — betesmarkers roll för svensk naturbetesflora

**Nästa steg:**
- Koppla till SMHI-meteorologidata → förklara anomalier (torka, översvämning)
- Utöka till hela Sverige → nationell bete-atlas
- Multi-årsanalys → identifiera trend (övergivning, förtätning)
- SNSA-finansierat uppföljningsprojekt för naturbetesmarker (SDL 2.0 → SDL 3.0 kontinuitet)

### Länkar
- [ImintEngine bete-showcase](https://digitalearth.se/case/)
- [ImintEngine/imint/analyzers/grazing.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/grazing.py)
- [LPIS öppna data hos Jordbruksverket](https://jordbruksverket.se)
- [NMD öppna data hos Naturvårdsverket](https://www.naturvardsverket.se/nmd)